In [ ]:
pip install ipywidgets

In [1]:
import ipywidgets as widgets
widgets.Dropdown(options=["Overexpression", "CRISPR"], description="Goal:")

Widget Javascript not detected.  It may not be installed or enabled properly. Reconnecting the current kernel may help.


In [1]:
pip install --upgrade notebook ipywidgets widgetsnbextension

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import ipywidgets as widgets
widgets.Dropdown(options=["CRISPR","RNAi"], description="Goal:")


Dropdown(description='Goal:', options=('CRISPR', 'RNAi'), value='CRISPR')

In [2]:
from IPython.display import clear_output

# Button to trigger filtering
button = widgets.Button(description="Suggest Vectors")
output = widgets.Output()

def on_button_clicked(b):
    clear_output(wait=True)
    display(goal_dd, system_dd, method_dd, button, output)

    # Filter dataframe
    filtered_df = df[
        (df["Goal"] == goal_dd.value) &
        (df["System"] == system_dd.value) &
        (df["Method"] == method_dd.value)
    ]

    with output:
        if not filtered_df.empty:
            print(f"🔍 Found {len(filtered_df)} matching vectors:\n")
            display(filtered_df[["Vector", "Promoter", "Tag", "Resistance", "Notes"]])
        else:
            print("⚠️ No exact matches. Try different filters.")

# Connect button to function
button.on_click(on_button_clicked)

# Display UI
display(goal_dd, system_dd, method_dd, button, output)


NameError: name 'goal_dd' is not defined

In [3]:
from IPython.display import clear_output

button = widgets.Button(description="Suggest Vectors")
output = widgets.Output()

def on_button_clicked(b):
    clear_output(wait=True)
    display(goal_dd, system_dd, method_dd, button, output)

    filtered_df = df[
        (df["Goal"] == goal_dd.value) &
        (df["System"] == system_dd.value) &
        (df["Method"] == method_dd.value)
    ]

    with output:
        if not filtered_df.empty:
            print(f"🔍 Found {len(filtered_df)} matching vectors:\n")
            display(filtered_df[["Vector", "Promoter", "Tag", "Resistance", "Notes"]])
        else:
            print("⚠️ No exact matches. Try different filters.")

button.on_click(on_button_clicked)
display(goal_dd, system_dd, method_dd, button, output)


NameError: name 'goal_dd' is not defined

In [4]:
from IPython.display import clear_output

button = widgets.Button(description="Suggest Vectors")
output = widgets.Output()

def on_button_clicked(b):
    clear_output(wait=True)
    display(goal_dd, system_dd, method_dd, button, output)

    filtered_df = df[
        (df["Goal"] == goal_dd.value) &
        (df["System"] == system_dd.value) &
        (df["Method"] == method_dd.value)
    ]

    with output:
        if not filtered_df.empty:
            print(f"🔍 Found {len(filtered_df)} matching vectors:\n")
            display(filtered_df[["Vector", "Promoter", "Tag", "Resistance", "Notes"]])
        else:
            print("⚠️ No exact matches. Try different filters.")

button.on_click(on_button_clicked)
display(goal_dd, system_dd, method_dd, button, output)


NameError: name 'goal_dd' is not defined

In [5]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load CSV
df = pd.read_csv("vector_data.csv")

# Create dropdowns
goal_dd = widgets.Dropdown(
    options=df["Goal"].dropna().unique().tolist(),
    description="Goal:"
)

system_dd = widgets.Dropdown(
    options=df["System"].dropna().unique().tolist(),
    description="System:"
)

method_dd = widgets.Dropdown(
    options=df["Method"].dropna().unique().tolist(),
    description="Method:"
)

# Button + Output
button = widgets.Button(description="Suggest Vectors")
output = widgets.Output()

def on_button_clicked(b):
    clear_output(wait=True)
    display(goal_dd, system_dd, method_dd, button, output)

    # Filter dataframe
    filtered_df = df[
        (df["Goal"] == goal_dd.value) &
        (df["System"] == system_dd.value) &
        (df["Method"] == method_dd.value)
    ]

    with output:
        if not filtered_df.empty:
            print(f"🔍 Found {len(filtered_df)} matching vectors:\n")
            display(filtered_df[["Vector", "Promoter", "Tag", "Resistance", "Notes"]])
        else:
            print("⚠️ No exact matches. Try different filters.")

# Connect button
button.on_click(on_button_clicked)

# Display UI
display(goal_dd, system_dd, method_dd, button, output)


Dropdown(description='Goal:', options=('Subcloning', 'Gateway Entry', 'Blunt-end', 'Toxic gene cloning', 'TA c…

Dropdown(description='System:', options=('Any', 'Dicot', 'Arabidopsis', 'Tomato', 'Monocot', 'Nicotiana'), val…

Dropdown(description='Method:', options=('Any', 'Agrobacterium'), value='Any')

Button(description='Suggest Vectors', style=ButtonStyle())

Output()

In [6]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load CSV
df = pd.read_csv("vector_data.csv")

# ---------- Helpers ----------
def clean(x):
    if pd.isna(x): return ""
    return str(x).strip().casefold()

dicots = {"arabidopsis","tomato","nicotiana","cotton","dicot"}
monocots = {"rice","maize","wheat","barley","monocot"}

def is_system_match(row_val, sel_val):
    rv, sv = clean(row_val), clean(sel_val)
    if sv in ("", "any"): return True
    if rv in ("", "any"): return True
    # species → family compatibility
    if sv in dicots:    return rv in dicots or rv == sv
    if sv in monocots:  return rv in monocots or rv == sv
    # family → family
    if sv == "dicot":   return rv in dicots
    if sv == "monocot": return rv in monocots
    # exact species match
    return rv == sv

def is_method_match(row_val, sel_val):
    rv, sv = clean(row_val), clean(sel_val)
    if sv in ("", "any"): return True
    if rv in ("", "any"): return True
    return rv == sv

def is_goal_match(row_val, sel_val):
    rv, sv = clean(row_val), clean(sel_val)
    if sv in ("", "any"): return True
    return rv == sv

def score_row(row, sel_goal, sel_system, sel_method):
    s = 0
    if is_goal_match(row["Goal"], sel_goal): s += 2
    if is_system_match(row["System"], sel_system): s += 1
    if is_method_match(row["Method"], sel_method): s += 1
    return s

# ---------- Widgets ----------
goals   = sorted(set(df["Goal"].dropna().unique().tolist()), key=str)
systems = sorted(set(df["System"].dropna().unique().tolist()), key=str)
methods = sorted(set(df["Method"].dropna().unique().tolist()), key=str)

goal_dd   = widgets.Dropdown(options=["Any"] + goals,   description="Goal:")
system_dd = widgets.Dropdown(options=["Any"] + systems, description="System:")
method_dd = widgets.Dropdown(options=["Any"] + methods, description="Method:")

button = widgets.Button(description="Suggest Vectors")
output = widgets.Output()

def filter_df():
    # mask each row with flexible rules
    mask = df.apply(
        lambda r: is_goal_match(r["Goal"], goal_dd.value)
                  and is_system_match(r["System"], system_dd.value)
                  and is_method_match(r["Method"], method_dd.value),
        axis=1
    )
    return df[mask]

def on_click(b):
    clear_output(wait=True)
    display(goal_dd, system_dd, method_dd, button, output)
    with output:
        print(f"Your picks → Goal: {goal_dd.value} | System: {system_dd.value} | Method: {method_dd.value}\n")
        hits = filter_df()
        if not hits.empty:
            display(hits[["Vector","Type","Goal","System","Method","Promoter","Tag","Resistance","Notes"]])
        else:
            print("⚠️ No exact matches. Here are the closest suggestions:\n")
            scored = df.copy()
            scored["Score"] = scored.apply(
                lambda r: score_row(r, goal_dd.value, system_dd.value, method_dd.value),
                axis=1
            )
            top = scored.sort_values("Score", ascending=False).head(6)
            display(top[["Vector","Type","Goal","System","Method","Promoter","Tag","Resistance","Notes","Score"]])

button.on_click(on_click)
display(goal_dd, system_dd, method_dd, button, output)


Dropdown(description='Goal:', options=('Any', 'Backbone', 'Blunt-end', 'CRISPR', 'Gateway Entry', 'Inducible E…

Dropdown(description='System:', options=('Any', 'Any', 'Arabidopsis', 'Dicot', 'Monocot', 'Nicotiana', 'Tomato…

Dropdown(description='Method:', options=('Any', 'Agrobacterium', 'Any'), value='Any')

Button(description='Suggest Vectors', style=ButtonStyle())

Output()

In [7]:
display(goal_dd, system_dd, method_dd, button, output)


Dropdown(description='Goal:', index=7, options=('Any', 'Backbone', 'Blunt-end', 'CRISPR', 'Gateway Entry', 'In…

Dropdown(description='System:', index=6, options=('Any', 'Any', 'Arabidopsis', 'Dicot', 'Monocot', 'Nicotiana'…

Dropdown(description='Method:', index=1, options=('Any', 'Agrobacterium', 'Any'), value='Agrobacterium')

Button(description='Suggest Vectors', style=ButtonStyle())

Output(outputs=({'name': 'stdout', 'text': 'Your picks → Goal: CRISPR | System: Tomato | Method: Agrobacterium…

In [8]:
pip install voila


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 396.8 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.6/181.6 KB 487.3 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
